# **Performance Different model of parsers**

In [ ]:
# Install dependencies:
!pip install openai spacy tqdm scikit-learn numpy sentence-transformers
!pip install stanza
!pip install trankit
!pip install flair


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 18.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trankit to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 38.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.4/773.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 46.3 MB/s eta 0:00:00
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941171 sha256=bf66fe391e3c996f5de0d9912dc18b90ee464e965093911dea24deed0c2163c8
  Stored in directory: /root/.cache/pip/wheels/3c/bc/9d/266e27289b9019680d65d9b608c37bff1eff565b001c97

In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
pip install adapter-transformers==3.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 12.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 6.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
!pip install supar
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 28.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import re
import ast
import torch
import spacy
import numpy as np
import time
import unicodedata
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from typing import List, Dict, Tuple, Set, Optional
from sentence_transformers import SentenceTransformer
from difflib import SequenceMatcher

# ── Lazy-loaded parser singletons (initialised once, reused everywhere) ──────
_stanza_pipeline  = None
_trankit_pipeline = None
_flair_parser     = None


# =============================================================================
# Configuration
# =============================================================================
api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    api_key = ""   # ← paste your key here if not using env var

TRAIN_DATA_PATH = "train.txt"
TEST_DATA_PATH  = "test.txt"


# =============================================================================
# Helper Functions  ── 100 % unchanged from your original
# =============================================================================

def read_line_examples_from_file(data_path: str) -> Tuple[List[str], List[list]]:
    """Reads data from file, each line is: sent####labels"""
    sents, labels = [], []
    with open(data_path, 'r', encoding='UTF-8') as fp:
        for line in fp:
            line = line.strip()
            if line != '' and '####' in line:
                words, tuples_str = line.split('####', 1)
                try:
                    tuples = ast.literal_eval(tuples_str)
                    sents.append(words)
                    labels.append(tuples)
                except (ValueError, SyntaxError):
                    print(f"Skipping malformed line: {line}")
    return sents, labels


def _normalize_text(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    s = re.sub(r"\s+", " ", s).strip()
    words_to_strip = ['a','an','the','of','in','on','at','for','to','with','by']
    words = s.split()
    while words and words[0]  in words_to_strip: words.pop(0)
    while words and words[-1] in words_to_strip: words.pop(-1)
    return " ".join(words)


def _opinion_sim(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
    normalize: bool = True,
    dedupe: bool = True,
    opinion_fuzzy_threshold: Optional[float] = None
) -> Dict[str, float]:
    """
    Computes micro precision/recall/F1 for quadruple lists with advanced matching options.
    A quadruple is [Aspect Term, Aspect Category, Sentiment Polarity, Opinion Term].
    ── unchanged from original ──
    """
    if len(all_preds) != len(all_golds):
        raise ValueError(f"Length mismatch: preds {len(all_preds)} vs golds {len(all_golds)}")

    total_tp = total_fp = total_fn = 0

    for preds, golds in zip(all_preds, all_golds):
        if normalize:
            g_list = [tuple(_normalize_text(x) for x in label) for label in golds]
            p_list = [tuple(_normalize_text(x) for x in label) for label in preds]
        else:
            g_list = [tuple(str(x) for x in label) for label in golds]
            p_list = [tuple(str(x) for x in label) for label in preds]

        if dedupe:
            g_list = list(dict.fromkeys(g_list))
            p_list = list(dict.fromkeys(p_list))

        matched_pred_idx = set()
        tp = 0

        for g in g_list:
            for pj, p in enumerate(p_list):
                if pj in matched_pred_idx:
                    continue
                non_opinion_match = (g[0] == p[0] and g[1] == p[1] and g[2] == p[2])
                if not non_opinion_match:
                    continue
                opinion_match = False
                if opinion_fuzzy_threshold is None:
                    if g[3] == p[3]:
                        opinion_match = True
                else:
                    if _opinion_sim(g[3], p[3]) >= opinion_fuzzy_threshold:
                        opinion_match = True
                if opinion_match:
                    tp += 1
                    matched_pred_idx.add(pj)
                    break

        fp = len(p_list) - tp
        fn = len(g_list) - tp
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1        = 2*(precision*recall)/(precision+recall) if (precision+recall) > 0 else 0

    print(f"Total True Positives: {total_tp}, False Positives: {total_fp}, False Negatives: {total_fn}")

    return {
        'precision': precision * 100,
        'recall':    recall    * 100,
        'f1':        f1        * 100
    }


# =============================================================================
# Parser loader helpers  ── one-time lazy initialisation per process
# =============================================================================

def _load_stanza():
    """Download + load Stanza English pipeline once."""
    global _stanza_pipeline
    if _stanza_pipeline is None:
        import stanza
        stanza.download('en', verbose=False)
        _stanza_pipeline = stanza.Pipeline(
            lang='en',
            processors='tokenize,mwt,pos,lemma,depparse',
            verbose=False
        )
    return _stanza_pipeline


def _load_trankit():
    """Download + load Trankit English pipeline once."""
    global _trankit_pipeline
    if _trankit_pipeline is None:
        from trankit import Pipeline as TrankitPipeline
        _trankit_pipeline = TrankitPipeline(
            'english',
            gpu=torch.cuda.is_available()
        )
    return _trankit_pipeline


def _load_flair():
    """Download + load Flair dependency parser once."""
    global _flair_parser
    if _flair_parser is None:
        from flair.models import DependencyParser
        # 'flair/dependency-parsing' is trained on UD English EWT (biaffine)
        _flair_parser = DependencyParser.load('flair/dependency-parsing')
    return _flair_parser


# =============================================================================
# DependencyTreeFormatter
#
# Drop-in replacement for your original class.
# ── All five original format methods are kept byte-for-byte identical when
#    parser_backend="spacy" (the default).
# ── Three new backends (stanza, trankit, flair) each produce the same five
#    output formats by converting their output into a unified token list first.
# =============================================================================

class DependencyTreeFormatter:
    """
    Handles different formats for dependency tree representations.

    New parameter
    -------------
    parser_backend : str
        "spacy"   - original SpaCy en_core_web_sm  (your baseline)
        "stanza"  - Stanford Stanza neural UD parser
        "trankit" - Trankit transformer-based UD parser
        "flair"   - Flair biaffine dependency parser
    """

    def __init__(self, nlp, parser_backend: str = "spacy"):
        self.nlp            = nlp                        # SpaCy model (always available)
        self.parser_backend = parser_backend.lower()

    # ------------------------------------------------------------------
    # Internal: parse sentence -> unified token list
    # Each dict has: id, text, lemma, pos, tag, dep, head_text, head_id
    # head_id is 1-based; 0 means ROOT
    # ------------------------------------------------------------------

    def _parse(self, sentence: str) -> List[Dict]:
        if self.parser_backend == "spacy":
            return self._parse_spacy(sentence)
        elif self.parser_backend == "stanza":
            return self._parse_stanza(sentence)
        elif self.parser_backend == "trankit":
            return self._parse_trankit(sentence)
        elif self.parser_backend == "flair":
            return self._parse_flair(sentence)
        else:
            raise ValueError(f"Unknown parser backend: '{self.parser_backend}'")

    def _parse_spacy(self, sentence: str) -> List[Dict]:
        doc = self.nlp(sentence)
        tokens = []
        for i, token in enumerate(doc, 1):
            tokens.append({
                "id":        i,
                "text":      token.text,
                "lemma":     token.lemma_,
                "pos":       token.pos_,
                "tag":       token.tag_,
                "dep":       token.dep_,
                "head_text": token.head.text,
                "head_id":   0 if token.dep_ == "ROOT" else token.head.i + 1,
                "_head_pos": token.head.pos_,
            })
        return tokens

    def _parse_stanza(self, sentence: str) -> List[Dict]:
        """
        Stanza word.head is 1-based; 0 = ROOT.
        """
        nlp = _load_stanza()
        doc = nlp(sentence)
        tokens = []
        for sent in doc.sentences:
            word_by_id = {w.id: w for w in sent.words}
            for word in sent.words:
                head_text = (
                    word_by_id[word.head].text
                    if word.head > 0 and word.head in word_by_id
                    else word.text
                )
                head_pos = (
                    word_by_id[word.head].upos
                    if word.head > 0 and word.head in word_by_id
                    else word.upos or "_"
                )
                tokens.append({
                    "id":        word.id,
                    "text":      word.text,
                    "lemma":     word.lemma  or word.text,
                    "pos":       word.upos   or "_",
                    "tag":       word.xpos   or "_",
                    "dep":       word.deprel or "_",
                    "head_text": head_text,
                    "head_id":   word.head,
                    "_head_pos": head_pos,
                })
        return tokens

    def _parse_trankit(self, sentence: str) -> List[Dict]:
        """
        Trankit returns a nested dict.  Multi-word tokens (id like "1-2") are skipped.
        """
        pipeline = _load_trankit()
        result   = pipeline(sentence)
        tokens   = []

        sentences = result.get("sentences", [result])
        for sent in sentences:
            word_list = [
                w for w in sent.get("tokens", [])
                if not (isinstance(w.get("id"), str) and "-" in str(w["id"]))
            ]
            id_to_word = {int(w["id"]): w for w in word_list}
            for word in word_list:
                wid      = int(word["id"])
                head_id  = int(word.get("head", 0))
                dep      = word.get("deprel", "_")
                head_w   = id_to_word.get(head_id)
                head_text = head_w["text"] if head_w else word["text"]
                head_pos  = head_w.get("upos", "_") if head_w else "_"
                tokens.append({
                    "id":        wid,
                    "text":      word.get("text",  "_"),
                    "lemma":     word.get("lemma", word.get("text", "_")),
                    "pos":       word.get("upos",  "_"),
                    "tag":       word.get("xpos",  "_"),
                    "dep":       dep,
                    "head_text": head_text,
                    "head_id":   head_id,
                    "_head_pos": head_pos,
                })
        return tokens

    def _parse_flair(self, sentence: str) -> List[Dict]:
        """
        Flair DependencyParser stores arc labels on each token.
        Label 'dep'  -> dependency relation
        Label 'head' -> 1-based head index (0 = ROOT)
        """
        from flair.data import Sentence as FlairSentence

        model      = _load_flair()
        flair_sent = FlairSentence(sentence)
        model.predict(flair_sent)

        tokens = []
        flair_tokens = flair_sent.tokens
        for i, token in enumerate(flair_tokens, 1):
            dep_val  = token.get_label("dep").value
            head_val = token.get_label("head").value
            head_id  = int(head_val) if head_val and head_val.lstrip("-").isdigit() else 0

            head_text = (
                flair_tokens[head_id - 1].text
                if 0 < head_id <= len(flair_tokens)
                else token.text
            )
            pos_val = token.get_label("upos").value or "_"
            tokens.append({
                "id":        i,
                "text":      token.text,
                "lemma":     token.text,   # Flair dep parser has no lemmatiser
                "pos":       pos_val,
                "tag":       "_",
                "dep":       dep_val or "_",
                "head_text": head_text,
                "head_id":   head_id,
                "_head_pos": "_",
            })
        return tokens

    # ------------------------------------------------------------------
    # Public format methods
    # When parser_backend="spacy" these are byte-for-byte identical to
    # your original implementation.
    # ------------------------------------------------------------------

    def format_spacy_default(self, sentence: str) -> str:
        """Original SpaCy format: token-head->dep"""
        if self.parser_backend == "spacy":
            # original code path, untouched
            doc = self.nlp(sentence)
            dep_lines = [f"{token.text}-{token.head.text}->{token.dep_}" for token in doc]
            return "\n".join(dep_lines)
        tokens = self._parse(sentence)
        return "\n".join(f"{t['text']}-{t['head_text']}->{t['dep']}" for t in tokens)

    def format_json(self, sentence: str) -> str:
        """JSON format with structured dependency information"""
        if self.parser_backend == "spacy":
            # original code path, untouched
            doc = self.nlp(sentence)
            dep_list = []
            for token in doc:
                dep_list.append({
                    "token":    token.text,
                    "pos":      token.pos_,
                    "dep":      token.dep_,
                    "head":     token.head.text,
                    "head_pos": token.head.pos_
                })
            return json.dumps(dep_list, indent=2)
        tokens = self._parse(sentence)
        dep_list = [{
            "token":    t["text"],
            "pos":      t["pos"],
            "dep":      t["dep"],
            "head":     t["head_text"],
            "head_pos": t["_head_pos"],
        } for t in tokens]
        return json.dumps(dep_list, indent=2)

    def format_conllu(self, sentence: str) -> str:
        """CoNLL-U format: ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS, MISC"""
        if self.parser_backend == "spacy":
            # original code path, untouched
            doc = self.nlp(sentence)
            lines = ["# text = " + sentence]
            for i, token in enumerate(doc, 1):
                head_idx = token.head.i + 1 if token.head.i != token.i else 0
                line = f"{i}\t{token.text}\t{token.lemma_}\t{token.pos_}\t{token.tag_}\t_\t{head_idx}\t{token.dep_}\t_\t_"
                lines.append(line)
            return "\n".join(lines)
        tokens = self._parse(sentence)
        lines  = ["# text = " + sentence]
        for t in tokens:
            lines.append(
                f"{t['id']}\t{t['text']}\t{t['lemma']}\t{t['pos']}\t{t['tag']}"
                f"\t_\t{t['head_id']}\t{t['dep']}\t_\t_"
            )
        return "\n".join(lines)

    def format_plain_text(self, sentence: str) -> str:
        """Plain text description format"""
        if self.parser_backend == "spacy":
            # original code path, untouched
            doc = self.nlp(sentence)
            lines = []
            for token in doc:
                if token.dep_ != "ROOT":
                    lines.append(f"'{token.text}' ({token.pos_}) depends on '{token.head.text}' via {token.dep_}")
                else:
                    lines.append(f"'{token.text}' ({token.pos_}) is the ROOT")
            return "\n".join(lines)
        tokens = self._parse(sentence)
        lines  = []
        for t in tokens:
            if t["dep"].upper() == "ROOT" or t["head_id"] == 0:
                lines.append(f"'{t['text']}' ({t['pos']}) is the ROOT")
            else:
                lines.append(f"'{t['text']}' ({t['pos']}) depends on '{t['head_text']}' via {t['dep']}")
        return "\n".join(lines)

    def format_hierarchical(self, sentence: str) -> str:
        """Hierarchical tree structure"""
        if self.parser_backend == "spacy":
            # original code path, untouched
            doc = self.nlp(sentence)
            def build_tree(token, level=0):
                indent = "  " * level
                result = f"{indent}|- {token.text} [{token.dep_}]\n"
                for child in token.children:
                    result += build_tree(child, level + 1)
                return result
            root = [token for token in doc if token.head == token][0]
            return build_tree(root)

        tokens   = self._parse(sentence)
        id_map   = {t["id"]: t for t in tokens}
        children: Dict[int, List[int]] = {t["id"]: [] for t in tokens}

        root_token = None
        for t in tokens:
            if t["dep"].upper() == "ROOT" or t["head_id"] == 0:
                root_token = t
            else:
                parent_id = t["head_id"]
                if parent_id in children:
                    children[parent_id].append(t["id"])

        if root_token is None and tokens:
            root_token = tokens[0]

        def build_tree(tid: int, level: int = 0) -> str:
            t      = id_map[tid]
            indent = "  " * level
            result = f"{indent}|- {t['text']} [{t['dep']}]\n"
            for child_id in children.get(tid, []):
                result += build_tree(child_id, level + 1)
            return result

        return build_tree(root_token["id"]) if root_token else ""


# =============================================================================
# GPT4o_ASQPExtractor
# ── All logic unchanged.  Only new parameter: parser_backend.
# =============================================================================

class GPT4o_ASQPExtractor:
    """
    An extractor class designed to evaluate the GPT-4o model via the OpenAI API.

    New parameter
    -------------
    parser_backend : str   which NLP parser generates the dependency tree
                           "spacy" | "stanza" | "trankit" | "flair"
    """
    def __init__(self, api_key: str,
                 dep_tree_format: str = "spacy_default",
                 parser_backend:  str = "spacy"):
        print(f"Initializing OpenAI client  |  parser={parser_backend}  |  format={dep_tree_format}")
        self.client          = OpenAI(api_key=api_key)
        self.nlp_sm          = spacy.load("en_core_web_sm")
        self.dep_formatter   = DependencyTreeFormatter(self.nlp_sm, parser_backend=parser_backend)
        self.dep_tree_format = dep_tree_format
        print("Client and tools initialized successfully!")

    def _get_dependency_tree(self, sentence: str) -> str:
        """Get dependency tree in the specified format"""
        if self.dep_tree_format == "spacy_default":
            return self.dep_formatter.format_spacy_default(sentence)
        elif self.dep_tree_format == "json":
            return self.dep_formatter.format_json(sentence)
        elif self.dep_tree_format == "conllu":
            return self.dep_formatter.format_conllu(sentence)
        elif self.dep_tree_format == "plain_text":
            return self.dep_formatter.format_plain_text(sentence)
        elif self.dep_tree_format == "hierarchical":
            return self.dep_formatter.format_hierarchical(sentence)
        else:
            raise ValueError(f"Unknown dependency tree format: {self.dep_tree_format}")

    def _create_messages(self, instruction: str, query: str,
                         dependency_tree: str, few_shot_examples: List[Dict]) -> List[Dict]:
        """Formats the prompt into the official OpenAI messages format."""
        system_content = instruction
        few_shot_text  = ""
        if few_shot_examples:
            few_shot_text = "### Examples\nHere are some examples to guide you:\n"
            for example in few_shot_examples:
                few_shot_text += f"\nSentence: {example['sentence']}\nLabels: {str(example['labels'])}\n"
            few_shot_text += "\n### Now, based on instruction and these examples, perform the task for the following sentence:\n"

        user_content  = f"{few_shot_text}Query: {query}\n"
        user_content += f"Dependency Tree: {dependency_tree}\n"
        user_content += "LABELS:"

        return [
            {"role": "system", "content": system_content},
            {"role": "user",   "content": user_content},
        ]

    def _parse_output(self, output_text: str) -> List[list]:
        """Robustly parse the model's string output into a Python list of lists."""
        try:
            data = json.loads(output_text)
            if isinstance(data, list) and all(isinstance(i, (list, tuple)) and len(i) == 4 for i in data):
                return data
        except Exception:
            pass

        start = output_text.find('[')
        if start == -1:
            return []

        stack, end = 0, -1
        for i in range(start, len(output_text)):
            if   output_text[i] == '[': stack += 1
            elif output_text[i] == ']':
                stack -= 1
                if stack == 0:
                    end = i
                    break
        if end == -1:
            return []

        list_str = output_text[start:end+1]
        try:
            parsed_list = ast.literal_eval(list_str)
            if isinstance(parsed_list, list) and all(isinstance(i, (list, tuple)) and len(i) == 4 for i in parsed_list):
                return parsed_list
        except Exception:
            return []
        return []

    def extract_quadruples(self, instruction: str, sentence: str,
                           few_shot_examples: List[Dict]) -> List[list]:
        """Gets a prediction from the GPT-4o API for a given sentence."""
        dependency_tree = self._get_dependency_tree(sentence)
        messages        = self._create_messages(instruction, sentence, dependency_tree, few_shot_examples)

        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                max_tokens=512,
                temperature=0.0
            )
            response_text = response.choices[0].message.content
            parsed = self._parse_output(response_text)
            if not parsed:
                print("DEBUG: could not parse model output:")
                print(response_text)
        except Exception as e:
            print(f"  [API Error]: {e}. Retrying after 10 seconds...")
            time.sleep(10)
            try:
                response = self.client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=messages,
                    max_tokens=512,
                    temperature=0.0
                )
                response_text = response.choices[0].message.content
                parsed = self._parse_output(response_text)
                if not parsed:
                    print("DEBUG: could not parse model output (retry):")
                    print(response_text)
            except Exception as e_retry:
                print(f"  [API Retry Failed]: {e_retry}. Returning empty list.")
                return []

        return parsed


# =============================================================================
# Evaluation runner
# ── Same logic as your original run_evaluation_for_format().
#    Only addition: parser_backend argument forwarded to the extractor.
# =============================================================================

def run_evaluation_for_format(
    dep_format:         str,
    parser_backend:     str,
    api_key:            str,
    train_sents:        List[str],
    train_labels:       List[list],
    test_sents:         List[str],
    test_labels:        List[list],
    instruction_prompt: str,
    num_few_shots:      int,
    train_embeddings:   np.ndarray,
    nlp_lg
) -> Dict:
    """
    Run evaluation for a specific (dep_format, parser_backend) combination.
    Logic is identical to the original run_evaluation_for_format().
    """
    run_id = f"{parser_backend}__{dep_format}"
    print(f"\n{'='*60}")
    print(f"EVALUATING  |  parser={parser_backend.upper()}  |  format={dep_format.upper()}")
    print(f"{'='*60}")

    extractor = GPT4o_ASQPExtractor(
        api_key=api_key,
        dep_tree_format=dep_format,
        parser_backend=parser_backend
    )

    def retrieve_few_shots(query_sentence: str, num_examples: int = 1) -> List[Dict]:
        if num_examples <= 0:
            return []
        query_embedding = nlp_lg(query_sentence).vector.reshape(1, -1)
        similarities    = cosine_similarity(query_embedding, train_embeddings).flatten()
        top_indices     = np.argsort(similarities)[-num_examples:][::-1]
        return [{"sentence": train_sents[idx], "labels": train_labels[idx]} for idx in top_indices]

    all_predictions   = []
    all_ground_truths = test_labels

    print(f"\n--- Starting Evaluation on {len(test_sents)} Test Samples ---")
    for i, sentence in enumerate(tqdm(test_sents)):
        few_shot_examples = retrieve_few_shots(sentence, num_examples=num_few_shots) if num_few_shots > 0 else []
        predicted_quads   = extractor.extract_quadruples(instruction_prompt, sentence, few_shot_examples)
        all_predictions.append(predicted_quads)

        if (i + 1) % 10 == 0:
            print(f"\n----- Sample {i+1}/{len(test_sents)} -----")
            print(f" Sentence:     {sentence}")
            print(f" Ground Truth: {all_ground_truths[i]}")
            print(f" Prediction:   {predicted_quads}")
            print("-" * 25)

        time.sleep(2)   # API rate limiting

    print("\n\nCalculating scores with advanced metrics...")
    metrics = calculate_metrics(
        all_preds=all_predictions,
        all_golds=all_ground_truths,
        normalize=True,
        dedupe=True,
        opinion_fuzzy_threshold=None
    )

    results_to_save = [
        {
            "sentence":     test_sents[i],
            "ground_truth": all_ground_truths[i],
            "prediction":   all_predictions[i]
        }
        for i in range(len(test_sents))
    ]
    filename = f"evaluation_results_{run_id}_{num_few_shots}shot.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results_to_save, f, indent=4)
    print(f"\nDetailed results saved to '{filename}'")

    return {
        "format":         dep_format,
        "parser_backend": parser_backend,
        "run_id":         run_id,
        "metrics":        metrics,
        "predictions":    all_predictions
    }


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    # ── Configuration ────────────────────────────────────────────────────────
    NUM_FEW_SHOTS = 1

    # Each tuple: (dep_tree_format, parser_backend)
    # ── You can freely add / remove combinations ─────────────────────────────
    # Available dep_tree_format values : "spacy_default", "json", "conllu",
    #                                    "plain_text", "hierarchical"
    # Available parser_backend values  : "spacy", "stanza", "trankit", "flair"
    EVALUATION_COMBINATIONS = [
        ── SpaCy  (your existing baseline)
        ("spacy_default", "spacy"),
        ("json",          "spacy"),
        ("conllu",        "spacy"),
        ("plain_text",    "spacy"),
        ("hierarchical",  "spacy"),

        ── Stanza  (Stanford neural UD parser)
        ("spacy_default", "stanza"),
        ("json",          "stanza"),
        ("conllu",        "stanza"),
        ("plain_text",    "stanza"),
        ("hierarchical",  "stanza"),

        # # ── Trankit  (transformer-based UD parser)
        # ("spacy_default", "trankit"),
        # ("json",          "trankit"),
        # ("conllu",        "trankit"),
        # ("plain_text",    "trankit"),
        # ("hierarchical",  "trankit"),

        # ── Flair  (biaffine dependency parser)
        # ("spacy_default", "flair"),
        # ("json",          "flair"),
        # ("conllu",        "flair"),
        # ("plain_text",    "flair"),
        # ("hierarchical",  "flair"),
    ]

    # ── Data Loading ─────────────────────────────────────────────────────────
    print("Loading datasets and building the few-shot retriever...")
    train_sents, train_labels = read_line_examples_from_file(TRAIN_DATA_PATH)
    test_sents,  test_labels  = read_line_examples_from_file(TEST_DATA_PATH)

    nlp_lg = spacy.load("en_core_web_lg")
    print("Building few-shot retriever index...")
    train_embeddings = np.array([nlp_lg(sent).vector for sent in tqdm(train_sents)])

    # ── Instruction Prompt  (unchanged from original) ─────────────────────────
    instruction_prompt = """
ROLE:
You are an expert ASQP (aspect sentiment quadruple prediction) annotation bot. Your processing must be precise, systematic, and strictly follow all rules.

TASK DEFINITION:
Your sole task is to extract all sentiment quadruples from a user-provided sentence.
Your response MUST be a valid Python list of lists, with no other text or explanations.
The output format for each quadruple is a list of four strings:
[<Aspect Term>, <Aspect Category>, <Sentiment Polarity>, <Opinion Term>]

ELEMENT DEFINITIONS:
1.  Aspect Term: The specific noun or phrase being reviewed.
    - It MUST be a term from the restaurant domain (e.g., "food", "service", "waiter").
    - Use the exact string "NULL" if the opinion is general or the aspect is implicit.
2.  Aspect Category: The general class for the Aspect Term.
    - You MUST choose from this predefined list only:
        - `ambience general`
        - `drinks prices`
        - `drinks quality`
        - `drinks style_options`
        - `food prices`
        - `food quality`
        - `food style_options`
        - `location general`
        - `restaurant general`
        - `restaurant miscellaneous`
        - `restaurant prices`
        - `service general`
3.  Sentiment: The polarity of the opinion.
    - It MUST be one of three strings: "positive", "negative", or "neutral".
4.  Opinion Term: The exact word(s) from the sentence expressing the sentiment.

EXTRACTION RULES:
1.  Concise Spans: The Opinion Term must be the most concise phrase that still captures the full sentiment. Do not include unnecessary words.
2.  Extract All Opinions: You must extract a separate quadruple for every distinct opinion expressed in the sentence.
3.  Implied Sentiment: Analyze comparative and conditional phrases to determine the implied sentiment.

STEP BY STEP PROCESS:
To ensure accuracy, think step-by-step before generating the final list:
1.  First, identify all opinion-expressing words/phrases in the sentence.
2.  Second, for each opinion, identify its specific aspect phrase. If none exist, that aspect is "NULL".
3.  Third, determine the sentiment polarity (positive, negative, neutral) and the correct aspect category from the provided list.
4.  Finally, construct the list of all quadruples according to all rules above.

FINAL OUTPUT INSTRUCTION:
Your final output MUST ONLY be a valid Python list of lists.
- Do not include explanations, reasoning, or markdown formatting.
- Do not include extra text before or after the list.
- Each inner list must contain exactly 4 strings.
"""

    # ── Run All Evaluations ───────────────────────────────────────────────────
    all_results = []

    for (dep_format, parser_backend) in EVALUATION_COMBINATIONS:
        result = run_evaluation_for_format(
            dep_format=dep_format,
            parser_backend=parser_backend,
            api_key=api_key,
            train_sents=train_sents,
            train_labels=train_labels,
            test_sents=test_sents,
            test_labels=test_labels,
            instruction_prompt=instruction_prompt,
            num_few_shots=NUM_FEW_SHOTS,
            train_embeddings=train_embeddings,
            nlp_lg=nlp_lg
        )
        all_results.append(result)

    # ── Final Comparison Report ───────────────────────────────────────────────
    print("\n\n" + "="*80)
    print("FINAL COMPARISON REPORT - ALL DEPENDENCY TREE FORMATS")
    print("="*80)
    print(f"\n{'Parser':<12} {'Format':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-" * 80)

    for result in all_results:
        metrics = result['metrics']
        print(
            f"{result['parser_backend']:<12} {result['format']:<20} "
            f"{metrics['precision']:>10.2f}%  {metrics['recall']:>10.2f}%  {metrics['f1']:>10.2f}%"
        )

    print("="*80)

    # Best overall
    best_result = max(all_results, key=lambda x: x['metrics']['f1'])
    print(
        f"\n BEST COMBINATION: parser={best_result['parser_backend'].upper()} | "
        f"format={best_result['format'].upper()} | "
        f"F1={best_result['metrics']['f1']:.2f}%"
    )

    # Best per parser
    print("\n BEST FORMAT PER PARSER:")
    from itertools import groupby
    from operator  import itemgetter
    for parser_name, group in groupby(
        sorted(all_results, key=itemgetter("parser_backend")),
        key=itemgetter("parser_backend")
    ):
        best = max(list(group), key=lambda x: x['metrics']['f1'])
        print(f"  {parser_name:<10} -> best format: {best['format']:<20} F1={best['metrics']['f1']:.2f}%")

    # Save consolidated comparison JSON  (same filename as original)
    comparison_data = {
        "num_few_shots":        NUM_FEW_SHOTS,
        "combinations_tested":  [f"{p}__{f}" for f, p in EVALUATION_COMBINATIONS],
        "results": [
            {
                "parser_backend": r["parser_backend"],
                "format":         r["format"],
                "run_id":         r["run_id"],
                "precision":      r["metrics"]["precision"],
                "recall":         r["metrics"]["recall"],
                "f1":             r["metrics"]["f1"],
            }
            for r in all_results
        ],
        "best_combination": best_result["run_id"],
        "best_f1":          best_result["metrics"]["f1"],
    }

    with open("dependency_tree_format_comparison.json", "w", encoding="utf-8") as f:
        json.dump(comparison_data, f, indent=4)

    print("\nComparison report saved to 'dependency_tree_format_comparison.json'")